In [10]:
import os, re, json, statistics
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

In [11]:
load_dotenv()   # reads the .env file

if os.getenv("OPENAI_API_KEY"):
    print("API key found")
else:
    print("No API key found. Check your .env file (it is needed for transcription).")

API key found


In [12]:
%pip install -q langchain-text-splitters pypdf openai tiktoken sentence-transformers matplotlib pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
from pathlib import Path

AUDIO_PATH = Path("The_Blueprint_For_Trustworthy_AI.m4a")

print("Current working folder:", Path.cwd())
if AUDIO_PATH.exists():
    print(f"Audio found: {AUDIO_PATH.name} ({AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("Audio NOT FOUND. Move the file into the folder shown above.")

Current working folder: /Users/pieteralbrecht/Desktop/IRONHACK/WEEK_3/WEEK_3_DAY_2/WEEK_3_DAY_2_LAB/lab-chunking-strategies
Audio found: The_Blueprint_For_Trustworthy_AI.m4a (30.2 MB)


In [14]:
import subprocess, glob

TRANSCRIPT_PATH = Path("transcript.txt")

def transcribe_audio(audio_path, out_path, model="whisper-1", piece_seconds=600):
    """Transcribe audio with the OpenAI API and save the text. Skips all work if out_path already exists."""
    out = Path(out_path)
    if out.exists() and out.stat().st_size > 0:
        print(f"{out} already exists -> skipping transcription")
        return out.read_text(encoding="utf-8")

    # 1) shrink and cut the audio: mono, 64 kbps mp3, 10-minute pieces
    Path("_pieces").mkdir(exist_ok=True)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(audio_path), "-ac", "1", "-b:a", "64k",
                    "-f", "segment", "-segment_time", str(piece_seconds), "-reset_timestamps", "1",
                    "_pieces/piece_%02d.mp3"], check=True)

    # 2) transcribe the pieces in order
    from openai import OpenAI
    client = OpenAI()                                  # uses OPENAI_API_KEY from the environment
    texts = []
    for f in sorted(glob.glob("_pieces/piece_*.mp3")):
        with open(f, "rb") as fh:
            texts.append(client.audio.transcriptions.create(model=model, file=fh).text)
        print(f, "done")

    text = " ".join(texts)
    out.write_text(text, encoding="utf-8")             # save to a local file
    return text

podcast_text = transcribe_audio(AUDIO_PATH, TRANSCRIPT_PATH)
print(f"Podcast transcript length: {len(podcast_text)} chars")
print(podcast_text[:200])

transcript.txt already exists -> skipping transcription
Podcast transcript length: 16461 chars
So imagine for a second you're driving across, I don't know, a massive suspension bridge. OK. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview t


In [15]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf")

def load_pdf(path=PDF_PATH):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

pdf_text = load_pdf()
print(f"PDF length: {len(pdf_text)} chars")
print(pdf_text[:200])

PDF length: 8692 chars
Ethics guidelines for trustworthy AI
On 8 April 2019, the High-Level Expert Group on AI presented Ethics Guidelines for
Trustworthy Artiﬁcial Intelligence. This followed the publication of the guideli


# For some reason the provided PDF got damaged, and threw an error. I downloaded the pdf from the website of the EU Commission. THe length varies though, but think it will work for now for this exercise.

In [18]:
documents = {"Podcast": podcast_text, "PDF": pdf_text}


def chunk_metrics(text, name, separator, chunk_size, chunk_overlap):
    chunks = CharacterTextSplitter(
        separator=separator,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    ).split_text(text)
    sizes = [len(chunk) for chunk in chunks]
    clean = sum(bool(chunk.strip()) and chunk.strip()[-1] in ".?!" for chunk in chunks)
    return {
        "Document": name,
        "Separator": repr(separator),
        "Size": chunk_size,
        "Overlap": chunk_overlap,
        "Chunks": len(chunks),
        "Average chars": round(sum(sizes) / len(sizes), 1) if sizes else 0,
        "Min chars": min(sizes, default=0),
        "Max chars": max(sizes, default=0),
        "Over size": sum(size > chunk_size for size in sizes),
        "Clean endings": f"{clean}/{len(chunks)} ({clean / len(chunks) * 100:.1f}%)" if chunks else "0/0 (0%)",
        "Paragraph breaks in text": text.count("\n\n"),
        "chunks": chunks,
    }


# Compare exact-size character splitting with paragraph-aware splitting.
separator_results = []
for document_name, text in documents.items():
    for separator in ("", "\n\n"):
        separator_results.append(chunk_metrics(text, document_name, separator, 500, 50))

print("Separator comparison (500 characters, 50 overlap):")
display(pd.DataFrame([{k: v for k, v in row.items() if k != "chunks"} for row in separator_results]))

# Check how changing chunk size and overlap affects clean sentence endings.
sweep_results = []
for document_name, text in documents.items():
    for chunk_size in (500, 1000, 2000):
        for overlap in (0, 50, 100):
            sweep_results.append(chunk_metrics(text, document_name, "", chunk_size, overlap))

print("Size/overlap sweep (separator=''):")
display(pd.DataFrame([{k: v for k, v in row.items() if k not in ("chunks", "Separator", "Paragraph breaks in text")} for row in sweep_results]))

# Inspect adjacent chunk boundaries; punctuation counts alone do not reveal mid-word cuts.
for document_name in documents:
    chunks = next(row["chunks"] for row in separator_results if row["Document"] == document_name and row["Separator"] == "''")
    print(f"\n{document_name} sample boundaries:")
    for index, chunk in enumerate(chunks[:3]):
        following = chunks[index + 1] if index + 1 < len(chunks) else ""
        print(f"Chunk {index + 1} end: {chunk[-50:]!r}")
        if following:
            print(f"Chunk {index + 2} start: {following[:50]!r}")

Separator comparison (500 characters, 50 overlap):


,Document,Separator,Size,Overlap,Chunks,Average chars,Min chars,Max chars,Over size,Clean endings,Paragraph breaks in text
0,Podcast,'',500,50,37,493.1,261,500,0,2/37 (5.4%),0
1,Podcast,'\n\n',500,50,1,16461.0,16461,16461,1,1/1 (100.0%),0
2,PDF,'',500,50,20,481.9,141,500,0,1/20 (5.0%),0
3,PDF,'\n\n',500,50,1,8692.0,8692,8692,1,0/1 (0.0%),0


Size/overlap sweep (separator=''):


,Document,Size,Overlap,Chunks,Average chars,Min chars,Max chars,Over size,Clean endings
0,Podcast,500,0,33,498.3,460,500,0,3/33 (9.1%)
1,Podcast,500,50,37,493.1,261,500,0,2/37 (5.4%)
2,Podcast,500,100,41,498.6,460,500,0,3/41 (7.3%)
3,Podcast,1000,0,17,967.9,460,1000,0,1/17 (5.9%)
4,Podcast,1000,50,18,961.3,311,1000,0,2/18 (11.1%)
5,Podcast,1000,100,19,960.8,261,1000,0,3/19 (15.8%)
6,Podcast,2000,0,9,1828.6,460,2000,0,1/9 (11.1%)
7,Podcast,2000,50,9,1873.3,861,2000,0,1/9 (11.1%)
8,Podcast,2000,100,9,1917.6,1261,2000,0,2/9 (22.2%)
9,PDF,500,0,18,482.6,192,500,0,1/18 (5.6%)



Podcast sample boundaries:
Chunk 1 end: ". Exactly. But now let's switch gears. Think about"
Chunk 2 start: "Exactly. But now let's switch gears. Think about t"
Chunk 2 end: ' arguably the Magna Carta for ethical computing, t'
Chunk 3 start: 'arguably the Magna Carta for ethical computing, th'
Chunk 3 end: "cally the Stone Age. We didn't have the generative"
Chunk 4 start: "cally the Stone Age. We didn't have the generative"

PDF sample boundaries:
Chunk 1 end: 'ethical principles and values\n(3) robust - both fr'
Chunk 2 start: 'ethical principles and values\n(3) robust - both fr'
Chunk 2 end: 's://ec.europa.eu/newsroom/dae/document.cfm?doc_id='
Chunk 3 start: 's://ec.europa.eu/newsroom/dae/document.cfm?doc_id='
Chunk 3 end: 'd=60428) | HU\n(https://ec.europa.eu/newsroom/dae/d'
Chunk 4 start: 'd=60428) | HU\n(https://ec.europa.eu/newsroom/dae/d'


# 1. does fixed size chuking break mid sentence?

# Yes it does, almost always.
# Podcast chunk 2 ends with " ethical computing, t" "the"
# PDF chunk 1 ends with "(3) robust - both fr" should be "from"
# PDF chunk 2 ends halfway through a URL

# Podcast 2/37 chunks end with "." ".?" or "!"
# 5.4%
# PDF 1/20 chunks
# 5%

# Changing size and overlap barely help. 2/9, 22.2%

# 2. How does it handle paragraph boundaries?
# It can't, because there are no paragraph boundaries in either text.
# an \n\n splitter may only split at a double break line. There is none here, so no split.

# 3. Which content type handles fixed-seze chunking better?

#	            Podcast	    PDF     
# Chunks	        37	        20
# Average	        493.1	    481.9
# Min	            261     	141
# Max	            500	        500
# Clean endings	    2/37 (5.4%)	1/20 (5.0%)

# Conclusion
# Fixed-size chunking ignores sentence and paragraph structure: at 500 characters with 50 overlap, only 5.4% of podcast chunks and 5.0% of PDF chunks end on ., ? or !, and words like “the” and “from” are cut in half. The default "\n\n" separator fails completely because both texts contain 0 paragraph breaks, so it returns one oversized chunk (16,461 chars for the podcast, 8,692 for the PDF). The podcast handles fixed-size chunking slightly better (up to 22.2% clean endings vs. at most 11.1% for the PDF), because its continuous sentences avoid the URLs and list fragments that the PDF cuts through.